# Data Generation — Job Matching ML

Generates synthetic but realistic data mirroring the production database schema.
All entities are domain-coherent (freelancers and jobs share skill vocabularies by domain).
Outcome labels are correlated with features so LightGBM can actually learn.

**Output:** CSV files in `data/` — one per entity + `ml_training_pairs.csv`

In [1]:
import numpy as np
import pandas as pd
import uuid
from datetime import date, timedelta
import random
import os

np.random.seed(42)
random.seed(42)

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Setup complete')

Setup complete


In [2]:
# Domain → skills and specialities mapping
DOMAIN_SKILLS = {
    'backend':    ['Python', 'Django', 'FastAPI', 'Flask', 'PostgreSQL', 'MySQL', 'Redis',
                   'REST API', 'Docker', 'Go', 'Node.js', 'Express.js', 'Microservices', 'System Design'],
    'frontend':   ['React', 'Vue.js', 'Angular', 'TypeScript', 'JavaScript', 'HTML', 'CSS',
                   'Next.js', 'Tailwind CSS', 'Webpack', 'Sass/SCSS', 'jQuery'],
    'fullstack':  ['React', 'Node.js', 'Python', 'PostgreSQL', 'REST API', 'Docker',
                   'TypeScript', 'JavaScript', 'HTML', 'CSS', 'MongoDB', 'Redis'],
    'mobile':     ['Flutter', 'React Native', 'iOS Development', 'Android Development',
                   'Dart', 'Kotlin', 'Swift', 'Firebase', 'REST API'],
    'ml_ai':      ['Python', 'Machine Learning', 'TensorFlow', 'PyTorch', 'scikit-learn',
                   'Pandas', 'NumPy', 'NLP', 'Computer Vision', 'LLM Integration', 'Data Science'],
    'design':     ['Figma', 'Adobe XD', 'Sketch', 'UI/UX Design', 'Graphic Design',
                   'Photoshop', 'Illustrator', 'Motion Design', 'HTML', 'CSS'],
    'devops':     ['Docker', 'Kubernetes', 'AWS', 'GCP', 'Azure', 'Terraform', 'Linux',
                   'CI/CD', 'GitHub Actions', 'Nginx', 'Python', 'Go'],
    'blockchain': ['Blockchain', 'Smart Contracts', 'Python', 'JavaScript', 'Solidity',
                   'Rust', 'Go'],
}

DOMAIN_SPECIALITIES = {
    'backend':    ['Backend Development', 'API Development', 'Database Architecture'],
    'frontend':   ['Frontend Development', 'Web Design', 'UI Development'],
    'fullstack':  ['Full-Stack Development', 'Web Development'],
    'mobile':     ['Mobile Development', 'App Development'],
    'ml_ai':      ['Machine Learning', 'Data Science', 'AI Engineering'],
    'design':     ['UI/UX Design', 'Graphic Design', 'Product Design'],
    'devops':     ['DevOps', 'Cloud Infrastructure', 'Site Reliability'],
    'blockchain': ['Blockchain Development', 'Web3', 'Smart Contract Development'],
}

ALL_DOMAINS = list(DOMAIN_SKILLS.keys())
LANGUAGES   = ['English', 'Indonesian', 'Mandarin', 'Spanish', 'French', 'German',
                'Japanese', 'Korean', 'Portuguese', 'Arabic', 'Hindi', 'Dutch']
print('Domain config ready:', ALL_DOMAINS)

Domain config ready: ['backend', 'frontend', 'fullstack', 'mobile', 'ml_ai', 'design', 'devops', 'blockchain']


In [3]:
# ── Reference tables ────────────────────────────────────────────────────────
SKILLS_RAW = [
    ('Python','hard_skill'),('JavaScript','hard_skill'),('TypeScript','hard_skill'),
    ('Java','hard_skill'),('Go','hard_skill'),('Rust','hard_skill'),
    ('C++','hard_skill'),('PHP','hard_skill'),('Ruby','hard_skill'),
    ('Swift','hard_skill'),('Kotlin','hard_skill'),('Dart','hard_skill'),
    ('C#','hard_skill'),('Scala','hard_skill'),('R','hard_skill'),
    ('React','hard_skill'),('Vue.js','hard_skill'),('Angular','hard_skill'),
    ('Next.js','hard_skill'),('Nuxt.js','hard_skill'),('HTML','hard_skill'),
    ('CSS','hard_skill'),('Tailwind CSS','tool'),('Bootstrap','tool'),
    ('Sass/SCSS','tool'),('jQuery','tool'),('Webpack','tool'),
    ('Django','hard_skill'),('FastAPI','hard_skill'),('Flask','hard_skill'),
    ('Node.js','hard_skill'),('Express.js','hard_skill'),('Laravel','hard_skill'),
    ('Spring Boot','hard_skill'),('Rails','hard_skill'),('ASP.NET','hard_skill'),
    ('PostgreSQL','hard_skill'),('MySQL','hard_skill'),('MongoDB','hard_skill'),
    ('Redis','hard_skill'),('Elasticsearch','hard_skill'),('SQLite','hard_skill'),
    ('Cassandra','hard_skill'),('DynamoDB','hard_skill'),('Supabase','tool'),
    ('Firebase','tool'),
    ('Docker','tool'),('Kubernetes','tool'),('AWS','tool'),('GCP','tool'),
    ('Azure','tool'),('GitHub Actions','tool'),('Terraform','tool'),
    ('Linux','tool'),('Nginx','tool'),('CI/CD','tool'),
    ('Machine Learning','hard_skill'),('TensorFlow','tool'),('PyTorch','tool'),
    ('scikit-learn','tool'),('Pandas','hard_skill'),('NumPy','hard_skill'),
    ('LLM Integration','hard_skill'),('Data Science','hard_skill'),
    ('Computer Vision','hard_skill'),('NLP','hard_skill'),
    ('Figma','tool'),('Adobe XD','tool'),('Sketch','tool'),
    ('UI/UX Design','hard_skill'),('Graphic Design','hard_skill'),
    ('Photoshop','tool'),('Illustrator','tool'),('Motion Design','hard_skill'),
    ('Flutter','hard_skill'),('React Native','hard_skill'),
    ('iOS Development','hard_skill'),('Android Development','hard_skill'),
    ('REST API','hard_skill'),('GraphQL','hard_skill'),('WebSocket','hard_skill'),
    ('Git','tool'),('gRPC','hard_skill'),('Microservices','hard_skill'),
    ('System Design','hard_skill'),('Blockchain','hard_skill'),
    ('Smart Contracts','hard_skill'),('Cybersecurity','hard_skill'),
    ('Solidity','hard_skill'),
    ('Communication','soft_skill'),('Leadership','soft_skill'),
    ('Problem Solving','soft_skill'),('Project Management','soft_skill'),
    ('Technical Writing','soft_skill'),('Agile','soft_skill'),('Scrum','soft_skill'),
]

skills_df = pd.DataFrame([{'skill_id': str(uuid.uuid4()), 'skill_name': n, 'skill_category': c}
                           for n, c in SKILLS_RAW])
skill_id_map = dict(zip(skills_df['skill_name'], skills_df['skill_id']))

all_spec_names = list(set(s for specs in DOMAIN_SPECIALITIES.values() for s in specs))
specialities_df = pd.DataFrame([{'speciality_id': str(uuid.uuid4()), 'speciality_name': n}
                                  for n in all_spec_names])
spec_id_map = dict(zip(specialities_df['speciality_name'], specialities_df['speciality_id']))

languages_df = pd.DataFrame([{'language_id': str(uuid.uuid4()), 'language_name': l,
                                'iso_code': l[:2].lower()} for l in LANGUAGES])
lang_id_map = dict(zip(languages_df['language_name'], languages_df['language_id']))

print(f'Skills: {len(skills_df)}, Specialities: {len(specialities_df)}, Languages: {len(languages_df)}')

Skills: 96, Specialities: 22, Languages: 12


In [4]:
# ── Freelancers ──────────────────────────────────────────────────────────────
N_FREELANCERS   = 700
COLD_START_N    = 175   # 25% cold start — no history

FIRST_NAMES = ['Alice','Bob','Charlie','Diana','Edward','Fatima','George','Hannah',
               'Ivan','Julia','Kevin','Lily','Marco','Nina','Oscar','Priya',
               'Quinn','Rachel','Samuel','Tanya','Umar','Victoria','William','Xena',
               'Yusuf','Zara','Ahmad','Budi','Citra','Dewi','Eko','Faisal','Gita',
               'Hendra','Indah','Joko','Kartika','Luthfi','Maya','Naufal','Putri']
LAST_NAMES  = ['Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis',
               'Santoso','Wijaya','Hidayat','Kusuma','Sari','Pratama','Utomo',
               'Lee','Kim','Chen','Wang','Kumar','Patel','Singh','Ahmed','Nakamura']

RATE_RANGES = {
    'backend':(20,80),'frontend':(15,70),'fullstack':(25,90),
    'mobile':(20,75),'ml_ai':(35,120),'design':(15,60),
    'devops':(30,100),'blockchain':(40,150),
}

freelancers, f_domains = [], []
for i in range(N_FREELANCERS):
    domain   = random.choice(ALL_DOMAINS)
    is_cold  = (i < COLD_START_N)
    exp_yrs  = 0 if is_cold else random.randint(1, 15)
    total_p  = 0 if is_cold else random.randint(0, 45)
    rmin, rmax = RATE_RANGES[domain]
    freelancers.append({
        'freelancer_id':  str(uuid.uuid4()),
        'user_id':        str(uuid.uuid4()),
        'full_name':      f'{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}',
        'bio':            f'{exp_yrs}-year {domain.replace("_"," ")} specialist.',
        'estimated_rate': round(random.uniform(rmin, rmax), 2),
        'rate_time':      'hourly',
        'rate_currency':  random.choices(['USD','USD','USD','IDR','EUR'], weights=[6,6,6,1,1])[0],
        'total_projects': total_p,
        'domain':         domain,
        'is_cold_start':  int(is_cold),
    })

freelancers_df = pd.DataFrame(freelancers)
print(f'Freelancers: {len(freelancers_df)}  ({COLD_START_N} cold start)')

Freelancers: 700  (175 cold start)


In [5]:
# ── Freelancer attributes ────────────────────────────────────────────────────
fl_skills_rows, fl_specs_rows, fl_langs_rows = [], [], []
work_exp_rows, education_rows, portfolio_rows = [], [], []

PROF_SKILL = ['expert','advanced','intermediate','beginner']
PROF_W     = [0.20, 0.40, 0.30, 0.10]

for _, f in freelancers_df.iterrows():
    fid    = f['freelancer_id']
    domain = f['domain']
    cold   = bool(f['is_cold_start'])

    # Skills: 3-9 primary + 1-2 cross-domain
    primary  = DOMAIN_SKILLS[domain]
    n_pick   = random.randint(3, min(9, len(primary)))
    chosen   = random.sample(primary, n_pick)
    other_sk = [s for d, sl in DOMAIN_SKILLS.items() if d != domain
                for s in sl if s not in chosen and s in skill_id_map]
    chosen  += random.sample(other_sk, min(random.randint(1, 2), len(other_sk)))
    seen = set()
    for sk in chosen:
        if sk in skill_id_map and sk not in seen:
            seen.add(sk)
            fl_skills_rows.append({
                'freelancer_skill_id': str(uuid.uuid4()),
                'freelancer_id': fid, 'skill_id': skill_id_map[sk],
                'skill_name': sk,
                'proficiency_level': random.choices(PROF_SKILL, weights=PROF_W)[0],
            })

    # Specialities: 1-3
    for j, sp in enumerate(random.sample(DOMAIN_SPECIALITIES[domain],
                                          random.randint(1, min(3, len(DOMAIN_SPECIALITIES[domain]))))):
        fl_specs_rows.append({
            'freelancer_speciality_id': str(uuid.uuid4()),
            'freelancer_id': fid, 'speciality_id': spec_id_map[sp],
            'speciality_name': sp, 'is_primary': int(j == 0),
        })

    # Languages: 1-3
    for j, lang in enumerate(random.sample(LANGUAGES, random.randint(1, 3))):
        fl_langs_rows.append({
            'freelancer_language_id': str(uuid.uuid4()),
            'freelancer_id': fid, 'language_id': lang_id_map[lang],
            'language_name': lang,
            'proficiency_level': 'native' if j == 0 else random.choice(['basic','conversational','fluent']),
        })

    if not cold:
        # Work experience: 1-4 jobs
        end = date.today()
        for j in range(random.randint(1, 4)):
            dur   = random.randint(6, 36)
            start = end - timedelta(days=dur * 30)
            work_exp_rows.append({
                'work_experience_id': str(uuid.uuid4()), 'freelancer_id': fid,
                'job_title':    ('Senior ' if j == 0 else '') + domain.replace('_',' ').title() + ' Engineer',
                'company_name': random.choice(['TechCorp','StartupXYZ','BigCo','Digital Agency',
                                               'Consultancy','Fintech Co','E-Commerce Ltd']),
                'start_date':   start.isoformat(),
                'end_date':     None if j == 0 else (end - timedelta(days=30)).isoformat(),
                'is_current':   int(j == 0),
                'description':  f'Built {domain.replace("_"," ")} solutions.',
            })
            end = start - timedelta(days=30)

        # Education: 55%
        if random.random() < 0.55:
            education_rows.append({
                'education_id': str(uuid.uuid4()), 'freelancer_id': fid,
                'institution_name': random.choice(['State University','Tech Institute','UI','ITS','ITB','NUS']),
                'degree': random.choice(['B.Sc Computer Science','B.Eng Software Engineering',
                                         'B.Sc Information Technology','M.Sc Computer Science']),
                'field_of_study': domain.replace('_',' ').title(),
                'start_date': '2018-08-01', 'end_date': '2022-06-01', 'is_current': 0,
            })

        # Portfolio: 55%
        if random.random() < 0.55:
            for _ in range(random.randint(1, 3)):
                portfolio_rows.append({
                    'portfolio_id': str(uuid.uuid4()), 'freelancer_id': fid,
                    'project_title': f'{domain.replace("_"," ").title()} Project',
                    'project_description': f'Built a {domain.replace("_"," ")} application.',
                })

fl_skills_df   = pd.DataFrame(fl_skills_rows)
fl_specs_df    = pd.DataFrame(fl_specs_rows)
fl_langs_df    = pd.DataFrame(fl_langs_rows)
work_exp_df    = pd.DataFrame(work_exp_rows)
education_df   = pd.DataFrame(education_rows)
portfolio_df   = pd.DataFrame(portfolio_rows)

print(f'freelancer_skills: {len(fl_skills_df)}')
print(f'freelancer_specialities: {len(fl_specs_df)}')
print(f'freelancer_languages: {len(fl_langs_df)}')
print(f'work_experience: {len(work_exp_df)}')
print(f'education: {len(education_df)}')
print(f'portfolio: {len(portfolio_df)}')

freelancer_skills: 5222
freelancer_specialities: 1306
freelancer_languages: 1424
work_experience: 1305
education: 288
portfolio: 585


In [6]:
# ── Job posts, roles, skills ─────────────────────────────────────────────────
N_JOBS = 600
BUDGET_RANGES = {
    'backend':(500,5000),'frontend':(300,4000),'fullstack':(800,8000),
    'mobile':(600,6000),'ml_ai':(1000,10000),'design':(300,3000),
    'devops':(800,8000),'blockchain':(1000,15000),
}

job_posts_rows, job_roles_rows, job_role_skills_rows = [], [], []

for _ in range(N_JOBS):
    domain   = random.choice(ALL_DOMAINS)
    jp_id    = str(uuid.uuid4())
    exp_lvl  = random.choices(['entry','intermediate','expert'], weights=[3,5,2])[0]
    bmin, bmax = BUDGET_RANGES[domain]

    job_posts_rows.append({
        'job_post_id':      jp_id,
        'client_id':        str(uuid.uuid4()),
        'job_title':        f'{domain.replace("_"," ").title()} Project',
        'job_description':  f'Need {exp_lvl} {domain.replace("_"," ")} developer.',
        'project_type':     random.choice(['individual','team']),
        'project_scope':    random.choice(['small','medium','large']),
        'experience_level': exp_lvl,
        'status':           random.choices(['active','closed','filled'], weights=[6,2,2])[0],
        'domain':           domain,
    })

    for role_idx in range(random.randint(1, 3)):
        jr_id = str(uuid.uuid4())
        job_roles_rows.append({
            'job_role_id':  jr_id, 'job_post_id': jp_id,
            'role_title':   domain.replace('_',' ').title() + (' Designer' if domain == 'design' else ' Engineer'),
            'role_budget':  round(random.uniform(bmin, bmax), 2),
            'budget_type':  random.choice(['fixed','negotiable']),
            'display_order': role_idx,
        })

        dom_sk   = DOMAIN_SKILLS[domain]
        n_req    = min(random.randint(2, 5), len(dom_sk))
        req_sk   = random.sample(dom_sk, n_req)
        other_sk = [s for d, sl in DOMAIN_SKILLS.items()
                    if d != domain for s in sl if s not in req_sk]
        pref_sk  = random.sample(other_sk, min(random.randint(0, 2), len(other_sk)))

        seen = set()
        for sk in req_sk:
            if sk in skill_id_map and sk not in seen:
                seen.add(sk)
                job_role_skills_rows.append({
                    'job_role_skill_id': str(uuid.uuid4()),
                    'job_role_id': jr_id, 'job_post_id': jp_id,
                    'skill_id': skill_id_map[sk], 'skill_name': sk,
                    'is_required': True, 'importance_level': 'required',
                    'job_domain': domain, 'exp_level': exp_lvl,
                })
        for sk in pref_sk:
            if sk in skill_id_map and sk not in seen:
                seen.add(sk)
                job_role_skills_rows.append({
                    'job_role_skill_id': str(uuid.uuid4()),
                    'job_role_id': jr_id, 'job_post_id': jp_id,
                    'skill_id': skill_id_map[sk], 'skill_name': sk,
                    'is_required': False,
                    'importance_level': random.choice(['preferred','nice_to_have']),
                    'job_domain': domain, 'exp_level': exp_lvl,
                })

job_posts_df      = pd.DataFrame(job_posts_rows)
job_roles_df      = pd.DataFrame(job_roles_rows)
job_role_skills_df = pd.DataFrame(job_role_skills_rows)

print(f'job_posts: {len(job_posts_df)}')
print(f'job_roles: {len(job_roles_df)}')
print(f'job_role_skills: {len(job_role_skills_df)}')

job_posts: 600
job_roles: 1204
job_role_skills: 5418


In [7]:
# ── Feature engineering helpers (used for both training pairs and simulation) ─
fl_skill_set   = fl_skills_df.groupby('freelancer_id')['skill_name'].apply(set).to_dict()
fl_lang_set    = fl_langs_df.groupby('freelancer_id')['language_name'].apply(set).to_dict()
fl_spec_set    = fl_specs_df.groupby('freelancer_id')['speciality_name'].apply(set).to_dict()
has_portfolio  = set(portfolio_df['freelancer_id'].unique()) if len(portfolio_df) else set()
work_exp_cnt   = work_exp_df.groupby('freelancer_id').size().to_dict() if len(work_exp_df) else {}

job_req_skills  = job_role_skills_df[job_role_skills_df['is_required']].groupby('job_post_id')['skill_name'].apply(set).to_dict()
job_pref_skills = job_role_skills_df[~job_role_skills_df['is_required']].groupby('job_post_id')['skill_name'].apply(set).to_dict()
job_exp_map     = job_posts_df.set_index('job_post_id')['experience_level'].to_dict()
job_domain_map  = job_posts_df.set_index('job_post_id')['domain'].to_dict()
job_budget_map  = job_roles_df.groupby('job_post_id')['role_budget'].max().to_dict()

EXP_ORD = {'entry': 0, 'intermediate': 1, 'expert': 2}

def infer_exp(total_p):
    if total_p == 0:  return 'entry'
    if total_p < 10:  return 'intermediate'
    return 'expert'

freelancers_df['inferred_exp'] = freelancers_df['total_projects'].apply(infer_exp)
fl_exp_map    = freelancers_df.set_index('freelancer_id')['inferred_exp'].to_dict()
fl_domain_map = freelancers_df.set_index('freelancer_id')['domain'].to_dict()
fl_rate_map   = freelancers_df.set_index('freelancer_id')['estimated_rate'].to_dict()
fl_projects   = freelancers_df.set_index('freelancer_id')['total_projects'].to_dict()
fl_cold_map   = freelancers_df.set_index('freelancer_id')['is_cold_start'].to_dict()

def compute_features(fid, jp_id):
    f_sk  = fl_skill_set.get(fid, set())
    j_req = job_req_skills.get(jp_id, set())
    j_prf = job_pref_skills.get(jp_id, set())

    req_match = len(f_sk & j_req)
    req_total = len(j_req)
    skill_pct  = req_match / req_total if req_total else 0
    pref_pct   = len(f_sk & j_prf) / len(j_prf) if j_prf else 0

    f_exp = EXP_ORD.get(fl_exp_map.get(fid, 'entry'), 0)
    j_exp = EXP_ORD.get(job_exp_map.get(jp_id, 'intermediate'), 1)
    exp_match = int(f_exp >= j_exp)
    exp_delta = int(f_exp - j_exp)

    f_rate     = fl_rate_map.get(fid, 30)
    j_budget   = job_budget_map.get(jp_id, 1000)
    f_monthly  = f_rate * 160
    rate_ok    = int(f_monthly <= j_budget)
    rate_ratio = float(min(f_monthly / j_budget, 3.0)) if j_budget else 1.5

    lang_match  = int(bool(fl_lang_set.get(fid, set()) & {'English','Indonesian'}))
    j_dom       = job_domain_map.get(jp_id, '')
    dom_specs   = set(DOMAIN_SPECIALITIES.get(j_dom, []))
    spec_match  = int(bool(fl_spec_set.get(fid, set()) & dom_specs))
    dom_match   = int(fl_domain_map.get(fid, '') == j_dom)

    # Simulated cosine similarity — correlated with skill/domain overlap + noise
    cosine = 0.45 * skill_pct + 0.25 * dom_match + 0.10 * spec_match + 0.20 * float(np.random.beta(2,3))
    cosine = float(np.clip(cosine, 0.04, 0.98))

    return {
        'cosine_sim':             cosine,
        'skill_overlap_pct':      skill_pct,
        'skill_required_matched': req_match,
        'skill_required_total':   req_total,
        'skill_preferred_pct':    pref_pct,
        'experience_level_match': exp_match,
        'exp_delta':              exp_delta,
        'rate_in_budget':         rate_ok,
        'rate_ratio':             rate_ratio,
        'language_match':         lang_match,
        'speciality_match':       spec_match,
        'domain_match':           dom_match,
        'has_portfolio':          int(fid in has_portfolio),
        'work_exp_count':         work_exp_cnt.get(fid, 0),
    }

print('Feature helpers ready')

Feature helpers ready


In [8]:
# ── Historical: proposals → contracts → ratings ──────────────────────────────
proposals_rows, contracts_rows, ratings_rows = [], [], []

warm_fids = list(freelancers_df[freelancers_df['is_cold_start'] == 0]['freelancer_id'])
closed_jobs = job_posts_df[job_posts_df['status'].isin(['closed','filled'])]

for _, jp in closed_jobs.iterrows():
    jp_id  = jp['job_post_id']
    j_dom  = jp['domain']
    client = jp['client_id']

    same  = [f for f in warm_fids if fl_domain_map.get(f) == j_dom]
    other = [f for f in warm_fids if fl_domain_map.get(f) != j_dom]

    n_prop = random.randint(5, 14)
    cands  = random.sample(same,  min(int(n_prop * 0.65), len(same)))
    cands += random.sample(other, min(n_prop - len(cands), len(other)))

    first_role = job_roles_df[job_roles_df['job_post_id'] == jp_id]
    if first_role.empty: continue
    role_row = first_role.iloc[0]

    for fid in cands:
        feat = compute_features(fid, jp_id)
        log_odds = (2.5 * feat['skill_overlap_pct'] + 2.0 * feat['cosine_sim'] +
                    1.5 * feat['experience_level_match'] + 0.8 * feat['rate_in_budget'] +
                    0.5 * feat['speciality_match'] + 0.3 * feat['has_portfolio'] - 1.8)
        acc_prob = 1 / (1 + np.exp(-log_odds))

        r = random.random()
        status = 'accepted' if r < acc_prob * 0.8 else ('withdrawn' if r < acc_prob * 0.8 + 0.1 else 'rejected')

        prop_id = str(uuid.uuid4())
        proposals_rows.append({
            'proposal_id': prop_id, 'job_post_id': jp_id,
            'job_role_id': str(role_row['job_role_id']), 'freelancer_id': fid,
            'proposed_budget': round(role_row['role_budget'] * random.uniform(0.8, 1.2), 2),
            'status': status,
        })

        if status == 'accepted':
            succ_p  = 0.35 + 0.50 * feat['skill_overlap_pct'] + 0.15 * feat['cosine_sim']
            c_stat  = random.choices(['completed','cancelled','active'], weights=[succ_p, 0.10, 0.15])[0]
            cont_id = str(uuid.uuid4())
            contracts_rows.append({
                'contract_id': cont_id, 'job_post_id': jp_id,
                'job_role_id': str(role_row['job_role_id']),
                'proposal_id': prop_id, 'freelancer_id': fid, 'client_id': client,
                'contract_title': f'Contract — {jp["job_title"]}',
                'agreed_budget': float(role_row['role_budget']),
                'payment_structure': random.choice(['full_payment','milestone_based']),
                'status': c_stat, 'start_date': '2024-01-01',
            })
            if c_stat == 'completed':
                base = 3.0 + 1.8 * feat['skill_overlap_pct'] + 0.2 * float(np.random.normal(0, 0.4))
                ovr  = round(max(1.0, min(5.0, base)), 2)
                ratings_rows.append({
                    'rating_id': str(uuid.uuid4()), 'contract_id': cont_id,
                    'client_id': client, 'freelancer_id': fid,
                    'communication_score':       random.randint(3, 5),
                    'result_quality_score':      min(5, max(1, int(ovr) + random.randint(-1, 1))),
                    'professionalism_score':     random.randint(3, 5),
                    'timeline_compliance_score': random.randint(2, 5),
                    'overall_rating': ovr,
                })

proposals_df = pd.DataFrame(proposals_rows)
contracts_df = pd.DataFrame(contracts_rows)
ratings_df   = pd.DataFrame(ratings_rows)

print(f'proposals: {len(proposals_df)}')
print(f'contracts: {len(contracts_df)}')
print(f'ratings:   {len(ratings_df)}')

proposals: 2206
contracts: 1422
ratings:   1018


In [9]:
# ── Performance ratings (aggregated) ────────────────────────────────────────
perf_rows = []
for fid in freelancers_df['freelancer_id']:
    fc = contracts_df[contracts_df['freelancer_id'] == fid] if len(contracts_df) else pd.DataFrame()
    fr = ratings_df[ratings_df['freelancer_id'] == fid]     if len(ratings_df)   else pd.DataFrame()
    total     = len(fc)
    completed = len(fc[fc['status'] == 'completed']) if total else 0
    if total == 0: continue
    avg_r    = fr['overall_rating'].mean()    if len(fr) else np.nan
    succ_r   = completed / total * 100
    perf_s   = (0.6 * (avg_r / 5 * 100) + 0.4 * succ_r) if not np.isnan(avg_r) else succ_r
    perf_rows.append({
        'performance_rating_id':     str(uuid.uuid4()),
        'freelancer_id':             fid,
        'overall_performance_score': round(perf_s, 2),
        'total_ratings_received':    len(fr),
        'average_result_quality':    round(fr['result_quality_score'].mean(), 2) if len(fr) else None,
        'success_rate':              round(succ_r, 2),
    })

perf_df = pd.DataFrame(perf_rows)
print(f'performance_ratings: {len(perf_df)}')

performance_ratings: 490


In [10]:
# ── ML training pairs ────────────────────────────────────────────────────────
perf_map = perf_df.set_index('freelancer_id')[['overall_performance_score','success_rate']].to_dict('index') \
           if len(perf_df) else {}

# Good outcome = contract completed + rating >= 4
good_cont_ids = set(ratings_df[ratings_df['overall_rating'] >= 4]['contract_id']) if len(ratings_df) else set()
good_prop_ids = set(contracts_df[contracts_df['contract_id'].isin(good_cont_ids)]['proposal_id']) \
                if len(contracts_df) else set()

ml_rows = []

def add_row(fid, jp_id, label):
    feat = compute_features(fid, jp_id)
    p    = perf_map.get(fid, {})
    feat.update({
        'performance_score': p.get('overall_performance_score', np.nan),
        'success_rate_hist': p.get('success_rate', np.nan),
        'total_projects':    int(fl_projects.get(fid, 0)),
        'is_cold_start':     int(fl_cold_map.get(fid, 0)),
        'freelancer_id':     fid, 'job_post_id': jp_id, 'label': label,
    })
    ml_rows.append(feat)

# Positive / negative from actual proposals
for _, prop in proposals_df.iterrows():
    add_row(prop['freelancer_id'], prop['job_post_id'],
            int(prop['proposal_id'] in good_prop_ids))

# Extra random non-proposer negatives (2:1 ratio)
existing_pairs = set(zip(proposals_df['freelancer_id'], proposals_df['job_post_id']))
all_jp_ids     = list(job_posts_df['job_post_id'])
target_neg     = min(len(ml_rows) * 2, 5000)
added, tries   = 0, 0

while added < target_neg and tries < target_neg * 12:
    tries += 1
    fid   = random.choice(warm_fids)
    jp_id = random.choice(all_jp_ids)
    if (fid, jp_id) not in existing_pairs:
        existing_pairs.add((fid, jp_id))
        add_row(fid, jp_id, 0)
        added += 1

# Also include some cold-start pairs (label=0 by default — no history = no success yet)
cold_fids = list(freelancers_df[freelancers_df['is_cold_start'] == 1]['freelancer_id'])
for fid in random.sample(cold_fids, min(400, len(cold_fids))):
    for jp_id in random.sample(all_jp_ids, min(3, len(all_jp_ids))):
        if (fid, jp_id) not in existing_pairs:
            existing_pairs.add((fid, jp_id))
            add_row(fid, jp_id, 0)

ml_df = pd.DataFrame(ml_rows)
print(f'ML training pairs: {len(ml_df)}')
print(f'Positive rate:     {ml_df["label"].mean():.3f}')
print(f'Cold start pairs:  {ml_df["is_cold_start"].sum()} ({ml_df["is_cold_start"].mean():.1%})')

ML training pairs: 7143
Positive rate:     0.059
Cold start pairs:  525 (7.3%)


In [11]:
# ── Save all CSVs ────────────────────────────────────────────────────────────
to_save = {
    'skills':                  skills_df,
    'specialities':            specialities_df,
    'languages':               languages_df,
    'freelancers':             freelancers_df,
    'freelancer_skills':       fl_skills_df,
    'freelancer_specialities': fl_specs_df,
    'freelancer_languages':    fl_langs_df,
    'work_experiences':        work_exp_df,
    'educations':              education_df,
    'portfolios':              portfolio_df,
    'job_posts':               job_posts_df,
    'job_roles':               job_roles_df,
    'job_role_skills':         job_role_skills_df,
    'proposals':               proposals_df,
    'contracts':               contracts_df,
    'ratings':                 ratings_df,
    'performance_ratings':     perf_df,
    'ml_training_pairs':       ml_df,
}

print('=== SAVING DATA ===')
for name, df in to_save.items():
    path = f'{DATA_DIR}/{name}.csv'
    df.to_csv(path, index=False)
    print(f'  {name:30s}  {len(df):6,d} rows  →  {path}')

print('\nAll data saved!')

=== SAVING DATA ===
  skills                              96 rows  →  data/skills.csv
  specialities                        22 rows  →  data/specialities.csv
  languages                           12 rows  →  data/languages.csv
  freelancers                        700 rows  →  data/freelancers.csv
  freelancer_skills                5,222 rows  →  data/freelancer_skills.csv
  freelancer_specialities          1,306 rows  →  data/freelancer_specialities.csv
  freelancer_languages             1,424 rows  →  data/freelancer_languages.csv
  work_experiences                 1,305 rows  →  data/work_experiences.csv
  educations                         288 rows  →  data/educations.csv
  portfolios                         585 rows  →  data/portfolios.csv
  job_posts                          600 rows  →  data/job_posts.csv
  job_roles                        1,204 rows  →  data/job_roles.csv
  job_role_skills                  5,418 rows  →  data/job_role_skills.csv
  proposals                      

  contracts                        1,422 rows  →  data/contracts.csv


  ratings                          1,018 rows  →  data/ratings.csv
  performance_ratings                490 rows  →  data/performance_ratings.csv
  ml_training_pairs                7,143 rows  →  data/ml_training_pairs.csv

All data saved!
